# 03 · From sessions to features
Count vectors for the classical models, ordered sequences for the deep models, and a chronological split.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import PARSED_DIR, RAW_DIR, RESULTS_DIR, SEQUENCES_DIR

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.edgecolor": "#888",
                     "axes.titleweight": "bold", "axes.titlesize": 11})
pd.set_option("display.max_colwidth", 120)

In [2]:
from src.features.count_vectors import EventCountVectorizer
from src.grouping.split import chronological_split
sessions = pd.read_parquet(SEQUENCES_DIR / "hdfs_sessions.parquet")
train, val, test = chronological_split(sessions)
vec = EventCountVectorizer().fit(train.events)
X = vec.transform(test.events.head(5))
pd.DataFrame(X.toarray(), columns=[f"E{e}" for e in vec.event_ids_] + ["UNSEEN"], index=test.session_id.head(5)).loc[:, lambda d: d.sum() > 0]

,E1,E2,E3,E4,E5,E17,E32,E33
session_id,,,,,,,,
blk_-1148420578838883691,3.0,1.0,3.0,3.0,3.0,3.0,3.0,0.0
blk_230214593119602688,3.0,1.0,3.0,3.0,3.0,3.0,3.0,0.0
blk_1989862932624214919,3.0,1.0,3.0,3.0,3.0,3.0,3.0,0.0
blk_5508364352111305966,3.0,1.0,3.0,3.0,3.0,3.0,3.0,0.0
blk_6618050731122867921,3.0,1.0,3.0,3.0,3.0,3.0,3.0,1.0


The ordered sequence of the first session (what the LSTM / Transformer see):

In [3]:
print(test.events.iloc[0])

[ 1  1  1  2  3  4  3  4  3  4  5  5  5 32 32 32 17 17 17]


## Chronological split

In [4]:
rows = []
for ds in ("hdfs", "bgl"):
    s = pd.read_parquet(SEQUENCES_DIR / f"{ds}_sessions.parquet")
    for name, part in zip(("train", "val", "test"), chronological_split(s)):
        rows.append({"dataset": ds.upper(), "split": name, "groups": len(part), "anomaly rate": f"{part.label.mean():.2%}",
                     "from": pd.to_datetime(part.start_time.min(), unit="s"), "to": pd.to_datetime(part.start_time.max(), unit="s")})
pd.DataFrame(rows)

,dataset,split,groups,anomaly rate,from,to
0,HDFS,train,345036,3.23%,2008-11-09 20:35:18,2008-11-11 04:17:26
1,HDFS,val,115012,3.48%,2008-11-11 04:17:26,2008-11-11 07:42:33
2,HDFS,test,115013,1.46%,2008-11-11 07:42:33,2008-11-11 11:04:13
3,BGL,train,2164,20.33%,2005-06-03 22:42:50,2005-10-11 14:42:50
4,BGL,val,721,12.34%,2005-10-11 15:42:50,2005-11-17 02:42:50
5,BGL,test,722,18.84%,2005-11-17 03:42:50,2006-01-04 15:42:50


## Unseen events
Templates that first appear after training are invisible to a vocabulary learned on training data. On BGL this is a large effect.

In [5]:
for ds in ("hdfs", "bgl"):
    s = pd.read_parquet(SEQUENCES_DIR / f"{ds}_sessions.parquet")
    tr, va, te = chronological_split(s)
    v = EventCountVectorizer().fit(tr.events)
    Xt = v.transform(te.events)
    unseen = np.asarray(Xt[:, v.unk_index_].todense()).ravel()
    print(f"{ds.upper()}: {int(unseen.sum()):,} of {int(Xt.sum()):,} test events unseen; "
          f"{(unseen > 0)[te.label == 1].mean():.1%} of anomalous vs {(unseen > 0)[te.label == 0].mean():.1%} of normal test groups contain one")

HDFS: 0 of 1,766,456 test events unseen; 0.0% of anomalous vs 0.0% of normal test groups contain one


BGL: 420,704 of 533,292 test events unseen; 96.3% of anomalous vs 84.3% of normal test groups contain one
